# <span style="color: #1F1DB5;">SPRINT 10 – Evaluación y Selección de Modelos

# <span style="color: #1F1DB5;">Notebook 1 – Métricas de Evaluación — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Entender por qué el accuracy puede ser engañoso en problemas desbalanceados.
- Diferenciar precision y recall usando la analogía médica.
- Calcular e interpretar el F1-score como balance entre precision y recall.
- Construir y leer una matriz de confusión para diagnosticar errores del modelo.
- Interpretar la curva ROC y el AUC como medida global de desempeño.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Entender por qué el **accuracy** puede ser engañoso en problemas desbalanceados.
- Diferenciar **precision** y **recall** usando la analogía médica.
- Calcular e interpretar el **F1-score** como balance entre precision y recall.
- Construir y leer una **matriz de confusión** para diagnosticar errores del modelo.
- Interpretar la **curva ROC** y el **AUC** como medida global de desempeño.
- Elegir la **métrica correcta** según el contexto de negocio.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
¿Por qué accuracy no es suficiente? | 15 min |
Precision y Recall: La analogía médica | 15 min |
F1-Score: El balance | 10 min |
Matriz de confusión: Visualización | 15 min |
Curva ROC y AUC | 15 min |
Elegir métrica por contexto de negocio | 10 min |
Actividad: Exposición por Equipos | 15 min |
Tips y buenas prácticas | 5 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### Un detector de fraude tiene 99.5% de accuracy, pero no detecta ni un solo fraude real. ¿Es un buen modelo? ¿Cómo medirías su verdadero desempeño?

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">¿Por qué accuracy no es suficiente? (15 mins)

Comencemos con un escenario que rompe la intuición:

**Caso del detector de fraude:** Tienes un dataset con 10,000 transacciones. Solo 50 son fraudes (0.5%). Si construyes un modelo que SIEMPRE dice "no es fraude" (sin siquiera mirar los datos), obtienes:

$$Accuracy = \frac{9950}{10000} = 99.5\%$$

¡99.5% de accuracy! Pero tu modelo es completamente inútil — no detecta ni un solo fraude.

Este es el **problema del desbalanceo de clases.** Cuando una clase domina el dataset, accuracy se infla artificialmente. Necesitamos métricas que midan lo que realmente importa para el negocio.

**Otros ejemplos donde accuracy engaña:**
- Detección de enfermedades raras (1 en 10,000 pacientes)
- Detección de spam (5% de correos son spam)
- Predicción de churn en clientes VIP (2% se van al mes)

Creemos un ejemplo concreto con código para demostrar el problema del accuracy:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Creamos un dataset desbalanceado (95% clase 0, 5% clase 1)
X, y = make_classification(n_samples=2000, n_features=10, n_informative=5,
                           weights=[0.95, 0.05], random_state=42, flip_y=0.01)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, 
                                                     random_state=42, stratify=y)

print(f"Distribución de clases en test:")
print(f"  Clase 0 (no fraude): {sum(y_test == 0)} ({sum(y_test == 0)/len(y_test)*100:.1f}%)")
print(f"  Clase 1 (fraude):    {sum(y_test == 1)} ({sum(y_test == 1)/len(y_test)*100:.1f}%)")

# "Modelo" que siempre predice la clase mayoritaria
y_pred_dummy = np.zeros_like(y_test)
print(f"\nModelo tonto (siempre predice 'no fraude'):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"  Fraudes detectados: 0 de {sum(y_test == 1)}")

# Modelo real
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f"\nModelo Logistic Regression:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"  Fraudes detectados: {sum((y_pred == 1) & (y_test == 1))} de {sum(y_test == 1)}")

Preguntas:

- ¿En qué tipo de problemas el accuracy SÍ es una buena métrica?

- ¿Qué tiene de malo un modelo que nunca detecta la clase minoritaria, aunque tenga alto accuracy?

- ¿Se te ocurre una situación laboral donde confiar en accuracy sea peligroso?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Precision y Recall: La analogía médica (15 mins)

Para entender precision y recall, usemos una analogía médica que nunca falla:

**Imagina una prueba de COVID:**

| Concepto | Analogía médica | Fórmula |
|---|---|---|
| **Precision** | De todos los que dieron POSITIVO, ¿cuántos realmente están enfermos? | TP / (TP + FP) |
| **Recall** | De todos los que REALMENTE están enfermos, ¿cuántos detectó la prueba? | TP / (TP + FN) |

**Terminología:**
- **TP (True Positive):** Dijiste "fraude" y SÍ era fraude. 
- **FP (False Positive):** Dijiste "fraude" pero NO era fraude. (Falsa alarma) 🚨
- **TN (True Negative):** Dijiste "no fraude" y NO era fraude. 
- **FN (False Negative):** Dijiste "no fraude" pero SÍ era fraude. (Se te escapó) 💀

**¿Cuándo priorizar cada una?**
- **Precision alta:** Cuando las falsas alarmas son costosas. Ej: enviar a cirugía por error.
- **Recall alto:** Cuando los falsos negativos son peligrosos. Ej: no detectar un cáncer.

Veamos precision y recall en acción con nuestro detector de fraude:

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("=" * 60)
print("REPORTE DE CLASIFICACIÓN - Detector de Fraude")
print("=" * 60)
print(f"\n{classification_report(y_test, y_pred, target_names=['No Fraude', 'Fraude'])}")

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"\nInterpretación para la clase 'Fraude':")
print(f"  Precision = {precision:.4f}")
print(f"    → De cada 100 alertas de fraude, {precision*100:.0f} realmente lo son.")
print(f"  Recall = {recall:.4f}")
print(f"    → De cada 100 fraudes reales, detectamos {recall*100:.0f}.")

print(f"\n💡 Trade-off: Si bajamos el umbral de decisión, detectamos más fraudes")
print(f"   (recall sube), pero también tenemos más falsas alarmas (precision baja).")

Preguntas:

- En un filtro de spam, ¿qué es peor: que un spam llegue a tu inbox (FN) o que un correo importante se vaya a spam (FP)?

- Para un auto autónomo detectando peatones, ¿priorizarías precision o recall?

- ¿Puede un modelo tener 100% precision y 0% recall? ¿Cómo?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">F1-Score: El balance (10 mins)

Cuando necesitas un solo número que balancee precision y recall, usamos el **F1-Score**. Es la media armónica de ambas:

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

¿Por qué media armónica y no promedio simple? Porque la media armónica **penaliza los extremos**. Si precision=1.0 y recall=0.01, el promedio simple sería 0.505 (parece bueno), pero F1 sería 0.02 (refleja la realidad: el modelo es pésimo en recall).

**Variantes:**
- **F1 macro:** Promedio simple del F1 de cada clase. Trata todas las clases igual.
- **F1 weighted:** Promedio ponderado por el número de muestras por clase.
- **F1 micro:** Calcula métricas globales sumando TP, FP, FN de todas las clases.

Comparemos los diferentes promedios de F1 para entender cuándo usar cada uno:

In [ ]:
# Comparar diferentes formas de calcular F1
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_binary = f1_score(y_test, y_pred, average='binary')

print("Diferentes formas de calcular F1:")
print(f"  F1 (binary - clase positiva): {f1_binary:.4f}")
print(f"  F1 (macro - promedio simple):  {f1_macro:.4f}")
print(f"  F1 (weighted - por tamaño):    {f1_weighted:.4f}")
print(f"\n→ Macro penaliza más cuando el modelo falla en la clase minoritaria")
print(f"→ Weighted da más peso a la clase mayoritaria (similar a accuracy)")

# Demostrar que media armónica penaliza extremos
print(f"\n--- Ejemplo del efecto de la media armónica ---")
p_extremo, r_extremo = 1.0, 0.01
promedio_simple = (p_extremo + r_extremo) / 2
f1_extremo = 2 * (p_extremo * r_extremo) / (p_extremo + r_extremo)
print(f"  Precision=1.0, Recall=0.01")
print(f"  Promedio simple: {promedio_simple:.3f} (engañoso)")
print(f"  F1 (media armónica): {f1_extremo:.3f} (realista)")

## <span style="color: #2826DE;">Matriz de confusión: Visualización (15 mins)

La matriz de confusión es la herramienta más completa para diagnosticar un clasificador. Muestra EXACTAMENTE dónde se equivoca el modelo.

**Lectura de la matriz (para clasificación binaria):**

| ' | Predicho: Negativo | Predicho: Positivo |
|---|---|---|
| **Real: Negativo** | TN (correcto) | FP (falsa alarma) |
| **Real: Positivo** | FN (se escapó) | TP (correcto) |

La diagonal principal (TN y TP) son predicciones correctas. Todo fuera de la diagonal son errores. Un modelo perfecto tiene solo valores en la diagonal.

Creemos un heatmap de la matriz de confusión para visualizar los errores de nuestro modelo:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Calcular la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Versión con conteos absolutos
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, 
                                display_labels=['No Fraude', 'Fraude'])
disp1.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusión (conteos)')

# Versión normalizada por fila (porcentaje de cada clase real)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, 
                                display_labels=['No Fraude', 'Fraude'])
disp2.plot(ax=axes[1], cmap='Oranges', colorbar=False, values_format='.2%')
axes[1].set_title('Matriz de Confusión (normalizada)')

plt.suptitle('Diagnóstico del Detector de Fraude', fontweight='bold')
plt.tight_layout()
plt.show()

# Interpretación
tn, fp, fn, tp = cm.ravel()
print(f"\nDesglose:")
print(f"  True Negatives  (correctos 'no fraude'): {tn}")
print(f"  False Positives (falsas alarmas):         {fp}")
print(f"  False Negatives (fraudes que se escaparon): {fn}")
print(f"  True Positives  (fraudes detectados):     {tp}")

Preguntas:

- ¿Qué cuadrante de la matriz quieres minimizar si eres un banco detectando fraude?

- ¿Y si eres un juez decidiendo si alguien va a prisión?

- ¿Cómo se ve la matriz de confusión de un modelo perfecto?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Curva ROC y AUC (15 mins)

La curva ROC (Receiver Operating Characteristic) nos da una visión global del desempeño de un clasificador **a través de todos los posibles umbrales de decisión**.

**Ejes de la curva ROC:**
- **Eje X:** False Positive Rate (FPR) = FP / (FP + TN) — proporción de negativos clasificados incorrectamente.
- **Eje Y:** True Positive Rate (TPR) = Recall = TP / (TP + FN)

**Interpretación:**
- La **línea diagonal** (de 0,0 a 1,1) representa un clasificador aleatorio (tirar una moneda).
- Un modelo perfecto pasa por el punto **(0, 1)** — detecta todos los positivos sin ningún falso positivo.
- **AUC** (Area Under the Curve): Un solo número entre 0.5 (aleatorio) y 1.0 (perfecto).

**Regla práctica:**
- AUC > 0.9: Excelente
- AUC 0.8 - 0.9: Bueno
- AUC 0.7 - 0.8: Aceptable
- AUC < 0.7: Pobre

Grafiquemos la curva ROC de nuestro modelo y calculemos el AUC:

In [ ]:
from sklearn.metrics import roc_curve, auc, RocCurveDisplay

# Obtener probabilidades (no solo la predicción binaria)
y_proba = model.predict_proba(X_test)[:, 1]

# Calcular la curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Aleatorio (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('Curva ROC')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall tradeoff en función del umbral
from sklearn.metrics import precision_recall_curve
prec_curve, rec_curve, thres_pr = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec_curve, prec_curve, color='green', lw=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Evaluación global del clasificador', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nAUC = {roc_auc:.4f}")
print(f"Interpretación: {'Excelente' if roc_auc > 0.9 else 'Bueno' if roc_auc > 0.8 else 'Aceptable'}")

Preguntas:

- ¿Qué significaría un AUC de exactamente 0.5?

- ¿Por qué la curva Precision-Recall es más informativa que ROC en datasets muy desbalanceados?

- Si mueves el umbral de decisión de 0.5 a 0.3, ¿qué le pasa al recall? ¿Y a la precision?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Elegir métrica por contexto de negocio (10 mins)

No existe una métrica universal. La elección depende 100% del **costo del error** en tu contexto de negocio.

| Contexto | Métrica prioritaria | Razonamiento |
|---|---|---|
| Detección de cáncer | **Recall** | No detectar un cáncer (FN) puede ser mortal. Mejor una biopsia innecesaria que un cáncer ignorado. |
| Filtro de spam | **Precision** | Perder un correo importante (FP) es peor que recibir un spam (FN). |
| Detección de fraude bancario | **F1 o Recall** | Un fraude no detectado cuesta dinero. Pero demasiadas alertas falsas saturan al equipo. |
| Recomendación de productos | **Precision** | Recomendar algo irrelevante (FP) molesta al usuario. |
| Sistema judicial | **Precision** | Condenar a un inocente (FP) es peor que dejar libre a un culpable (FN). |
| Detección de incendios | **Recall** | No detectar un incendio (FN) puede ser catastrófico. |

**Regla general:** Pregúntate "¿qué error me cuesta más?" — si es el FN → prioriza recall; si es el FP → prioriza precision.

## <span style="color: #2826DE;">Actividad: Exposición por Equipos (15 mins)

Cada equipo preparará una mini-presentación de 3 minutos sobre un subtema asignado. El objetivo es que practiquen explicar conceptos técnicos de forma clara y concisa.

### Instrucciones:

Cada equipo tendrá **3 minutos** para explicar una métrica al resto del grupo:

| Equipo | Métrica asignada | Debe incluir |
|---|---|---|
| Equipo 1 | Precision | Definición, fórmula, ejemplo real, cuándo priorizarla |
| Equipo 2 | Recall | Definición, fórmula, ejemplo real, cuándo priorizarla |
| Equipo 3 | F1-Score | Definición, fórmula, por qué media armónica, ejemplo |
| Equipo 4 | Matriz de confusión | Cómo leerla, qué cuadrante importa en fraude vs spam |
| Equipo 5 | ROC/AUC | Qué mide, cómo interpretar, cuándo AUC es engañoso |

**Criterios de evaluación:**
- Claridad de la explicación (¿la entendería alguien sin contexto?)
- Ejemplo del mundo real (no solo repetir la definición)
- Conexión con el contexto de negocio

**Tiempo total:** 3 min por equipo + 2 min de preguntas del grupo.

## <span style="color: #2826DE;">Tips y buenas prácticas

- **Siempre empieza** preguntando "¿cuál es el costo del error?" antes de elegir una métrica.
- En datasets desbalanceados, usa F1 macro o la curva Precision-Recall en lugar de ROC.
- `classification_report` de sklearn te da todo de un vistazo — úsalo siempre.
- Cuando presentes resultados a stakeholders no técnicos, traduce métricas a impacto de negocio: "De 1000 fraudes, nuestro modelo detecta 850" es más claro que "recall = 0.85".
- El umbral de decisión (0.5 por defecto) es un hiperparámetro que puedes optimizar según el contexto.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ Un modelo de detección de fraude tiene 99% accuracy pero 0% recall. ¿Qué está pasando?

- El modelo es excelente y no necesita mejoras
- El dataset está balanceado y el modelo funciona bien
- El modelo predice siempre "no fraude" en un dataset desbalanceado 
- El modelo tiene overfitting severo


2️⃣ ¿Qué mide el Recall?

- De todos los que marqué positivos, ¿cuántos lo eran realmente?
- De todos los que realmente son positivos, ¿cuántos detecté? 
- El porcentaje total de aciertos del modelo
- La proporción de falsos positivos


3️⃣ Para un sistema de detección de incendios, ¿qué métrica priorizarías?

- Accuracy
- Precision
- Recall 
- Specificity


4️⃣ ¿Qué significa un AUC de 0.5?

- El modelo es perfecto
- El modelo es tan bueno como tirar una moneda al azar 
- El modelo siempre acierta la clase positiva
- El modelo tiene precisión perfecta


5️⃣ La media armónica del F1-Score sirve para:

- Dar más peso a la clase mayoritaria
- Penalizar cuando precision o recall son extremadamente bajos 
- Calcular el promedio simple entre precision y recall
- Ignorar los falsos negativos


6️⃣ En la matriz de confusión, un False Negative significa:

- El modelo dijo "positivo" pero era negativo
- El modelo dijo "negativo" pero era positivo 
- El modelo acertó la predicción negativa
- El modelo acertó la predicción positiva